# Transformer car recommendation

Notebook version of the recommendation logic in `app.py`. It loads the car dataset, applies the same price/year/seats/brand/body-type filters, ranks matches with the `paraphrase-multilingual-MiniLM-L12-v2` Sentence Transformer model and displays the top five results.

Run this notebook with the repository's shared `requirements.txt` installed. Set the notebook working directory to `sources/Transformer` before running the cells. The model is downloaded from Hugging Face on its first use, so an internet connection is required.

In [2]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from IPython.display import display

DATASET_PATH = (Path.cwd().parent / "Dataset" / "car-price.csv").resolve()
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MAX_CANDIDATES = 500

if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. Run the notebook from sources/Transformer."
    )

print(f"Dataset: {DATASET_PATH}")

Dataset: D:\FSB-MSA\PPR501\PPR501-car-analysis\sources\Dataset\car-price.csv


In [3]:
def convert_price(price):
    if pd.isna(price):
        return np.nan

    price = str(price).strip()
    ty = re.search(r"([\d.,]+)\s*Tỷ", price, re.IGNORECASE)
    trieu = re.search(r"([\d.,]+)\s*Triệu", price, re.IGNORECASE)

    if not ty and not trieu:
        return np.nan

    ty_value = float(ty.group(1).replace(",", ".")) if ty else 0
    trieu_value = float(trieu.group(1).replace(",", ".")) if trieu else 0
    return ty_value * 1000 + trieu_value


def extract_year(question):
    match = re.search(r"\b(19\d{2}|20\d{2})\b", question)
    return int(match.group(1)) if match else None


def extract_seats(question):
    match = re.search(
        r"\b(\d+)\s*(?:chỗ(?:\s+ngồi)?|ghế)\b", question, re.IGNORECASE
    )
    return int(match.group(1)) if match else None


def find_column(df, names):
    normalized = {str(col).strip().lower(): col for col in df.columns}
    for name in names:
        key = name.lower()
        if key in normalized:
            return normalized[key]
    return None


def extract_brand(question, dataset):
    if not find_column(dataset, ["Name"]):
        return None

    brands = [
        "Mercedes-Benz", "Mercedes Benz", "Mitsubishi", "Volkswagen",
        "Land Rover", "VinFast", "Toyota", "Hyundai", "Honda", "Kia",
        "Mazda", "Ford", "BMW", "Audi", "Lexus", "Porsche", "Peugeot",
        "Volvo", "Subaru", "Isuzu", "Suzuki", "Nissan", "Chevrolet", "MG", "BYD",
    ]
    question_lower = question.casefold()
    for brand in sorted(brands, key=len, reverse=True):
        if brand.casefold() in question_lower:
            return brand
    return None


def extract_body_type(question):
    body_types = ["SUV", "Sedan", "MPV", "Hatchback", "Pickup", "Bán tải", "Crossover"]
    question_lower = question.casefold()
    for body_type in body_types:
        if body_type.casefold() in question_lower:
            return body_type
    return None


def extract_price_condition(question):
    match = re.search(
        r"(\d+(?:[.,]\d+)?)\s*(tỷ|triệu|tr|million)", question, re.IGNORECASE
    )
    if not match:
        return None, None

    amount = float(match.group(1).replace(",", "."))
    unit = match.group(2).lower()
    price = amount * 1000 if unit == "tỷ" else amount
    before = question[:match.start()].casefold()
    mode = "min" if re.search(r"(trên|hơn|ít nhất|từ)\s*$", before) else "max"
    return price, mode


def build_description(row, columns):
    parts = []
    for col in columns:
        value = row[col]
        if pd.notna(value):
            text = str(value).strip()
            if text:
                parts.append(f"{col}: {text}")
    return ". ".join(parts)

In [4]:
dataset = pd.read_csv(DATASET_PATH)
if "Price" not in dataset.columns:
    raise ValueError("Dataset must contain Price column")

dataset["Price_Million"] = dataset["Price"].apply(convert_price)
dataset = dataset.dropna(subset=["Price_Million"])
dataset = dataset[dataset["Price_Million"] > 0].copy()

model = SentenceTransformer(MODEL_NAME)
print(f"Loaded {len(dataset):,} cars with valid prices.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4432.81it/s]


Loaded 15,442 cars with valid prices.


In [5]:
def recommend_cars(question, dataset, model):
    price, price_mode = extract_price_condition(question)
    year = extract_year(question)
    seats = extract_seats(question)
    brand = extract_brand(question, dataset)
    body_type = extract_body_type(question)
    candidates = dataset.copy()

    if price is not None:
        if price_mode == "min":
            candidates = candidates[candidates["Price_Million"] >= price].copy()
        else:
            candidates = candidates[candidates["Price_Million"] <= price].copy()

    year_col = find_column(candidates, ["Năm sản xuất"])
    if year is not None and year_col:
        year_values = pd.to_numeric(candidates[year_col], errors="coerce")
        candidates = candidates[year_values == year].copy()

    seats_col = find_column(candidates, ["Số chỗ ngồi", "Số chỗ"])
    if seats is not None and seats_col:
        seat_values = pd.to_numeric(
            candidates[seats_col].astype(str).str.extract(r"(\d+)")[0],
            errors="coerce",
        )
        candidates = candidates[seat_values == seats].copy()

    name_col = find_column(candidates, ["Name"])
    if brand and name_col:
        candidates = candidates[
            candidates[name_col].astype(str).str.contains(
                re.escape(brand), case=False, na=False
            )
        ].copy()

    body_col = find_column(candidates, ["Kiểu dáng"])
    if body_type and body_col:
        candidates = candidates[
            candidates[body_col].astype(str).str.contains(
                re.escape(body_type), case=False, na=False
            )
        ].copy()

    if candidates.empty:
        return price, price_mode, pd.DataFrame()

    car_columns = [
        "Name", "Năm sản xuất", "Số chỗ ngồi", "Kiểu dáng", "Số Km đã đi",
        "Tình trạng", "Xuất xứ", "Động cơ", "Màu ngoại thất",
        "Màu nội thất", "Description",
    ]
    available_columns = []
    for name in car_columns:
        col = find_column(candidates, [name])
        if col and col not in available_columns:
            available_columns.append(col)

    if not available_columns:
        return price, price_mode, pd.DataFrame()

    year_col = find_column(candidates, ["Năm sản xuất"])
    if year_col:
        candidates["_Year"] = pd.to_numeric(candidates[year_col], errors="coerce")
        candidates = candidates.sort_values("_Year", ascending=False)

    candidates = candidates.head(MAX_CANDIDATES).copy()
    candidates["Description"] = candidates.apply(
        lambda row: build_description(row, available_columns), axis=1
    )
    candidates = candidates[candidates["Description"].str.len() > 0].copy()
    if candidates.empty:
        return price, price_mode, pd.DataFrame()

    question_embedding = model.encode([question], normalize_embeddings=True)
    car_embeddings = model.encode(
        candidates["Description"].tolist(),
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=False,
    )
    candidates["Similarity"] = car_embeddings @ question_embedding[0]
    candidates = candidates.sort_values("Similarity", ascending=False)
    candidates = candidates.drop_duplicates(
        subset=["Name"] if "Name" in candidates.columns else ["Description"]
    )
    return price, price_mode, candidates.head(5)

## Ask for recommendations

Run the next cell and enter a question in Vietnamese. Price values are interpreted in million VND (for example, `dưới 600 triệu` or `trên 1 tỷ`). You can include a year, seat count, brand, and body type. Re-run the cell to ask another question.

In [6]:
question = input("Nhập nhu cầu mua xe (ví dụ: Toyota 7 chỗ năm 2020 dưới 600 triệu): ").strip()
if not question:
    raise ValueError("Please enter a car search question.")

budget, price_mode, results = recommend_cars(question, dataset, model)
if results.empty:
    if budget is not None:
        print(f"Không tìm thấy xe phù hợp với ngân sách {budget:,.0f} triệu VND và các điều kiện đã nhập.")
    else:
        print("Không tìm thấy xe phù hợp. Hãy thử thay đổi hãng xe hoặc năm sản xuất.")
else:
    if budget is not None:
        operator = "từ" if price_mode == "min" else "không vượt"
        print(f"Tìm thấy xe có giá {operator} {budget:,.0f} triệu VND.")
    else:
        print("Đã tìm thấy các xe phù hợp với nhu cầu.")

    print("Kết quả được xếp hạng theo độ tương đồng ngữ nghĩa; thông tin phụ thuộc vào dataset.")
    name_col = find_column(results, ["Name", "Tên xe", "Dòng xe", "Model", "Hãng xe"])
    results = results.copy()
    results.insert(0, "Rank", range(1, len(results) + 1))
    if name_col:
        results = results.rename(columns={name_col: "Car"})
    display_columns = [
        col for col in [
            "Rank", "Car", "Price_Million", "Similarity", "Năm sản xuất",
            "Số chỗ ngồi", "Kiểu dáng", "Description",
        ] if col in results.columns
    ]
    display(results[display_columns].rename(columns={"Price_Million": "Price (million VND)"}))

Đã tìm thấy các xe phù hợp với nhu cầu.
Kết quả được xếp hạng theo độ tương đồng ngữ nghĩa; thông tin phụ thuộc vào dataset.


,Rank,Car,Price (million VND),Similarity,Năm sản xuất,Số chỗ ngồi,Kiểu dáng,Description
1791,1,Toyota Fortuner 2.4G 4x2 AT Legender - 2021,1036.0,0.467922,2021.0,7 chỗ,SUV,Name: Toyota Fortuner 2.4G 4x2 AT Legender - 2...
7425,2,Toyota Fortuner 2.8V 4x4 AT Legender - 2021,1150.0,0.466549,2021.0,7 chỗ,SUV,Name: Toyota Fortuner 2.8V 4x4 AT Legender - 2...
3901,3,Toyota Land Cruiser 3.5 V6 - 2023,5139.0,0.465246,2023.0,7 chỗ,SUV,Name: Toyota Land Cruiser 3.5 V6 - 2023. Năm s...
2548,4,Toyota Fortuner Legender 2.8L 4x4 AT - 2022,1290.0,0.463093,2022.0,7 chỗ,SUV,Name: Toyota Fortuner Legender 2.8L 4x4 AT - 2...
2072,5,Toyota Fortuner 2.4G 4x2 MT - 2021,795.0,0.459417,2021.0,7 chỗ,SUV,Name: Toyota Fortuner 2.4G 4x2 MT - 2021. Năm ...
